# PCR — Priority-Conditioned Retention (test-time, Mamba-790m)
**Importance-Aware Capacity Allocation in a *pretrained* continuous-state SSM**

이 노트북은 *학습 없이* 사전학습 Mamba-790m에서 retention을 추론 시점에 제어한다.
원래 제안서의 "학습된 [P1/P2/P3] grammar + d_state 용량법칙 + 3-way"가 아니라,
**실제로 검증된 test-time 기계해석 프로그램**을 처음부터 재현한다.

- 솔직한 스코프: **790m 단일 모델**. 1.4b 비재현·자연어 fragile은 limitation.
- 두 레버: (1) 생존 = Δ 게이트, (2) 정밀도 = write 양자화.
- T4에서 수십 분. `mamba-ssm`/`causal-conv1d`는 **설치하지 않는다**(slow path 강제 → Δ 훅 가능).

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Jun 20 03:02:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# [설치] fast path 막아서 slow path 강제 → dt_proj/conv1d 훅 가능
!pip -q install "transformers>=4.39" accelerate
import torch, random, math
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

torch 2.11.0+cu128 | cuda True


In [3]:
# [로드] 사전학습 790m, fp32, freeze
from transformers import AutoTokenizer, MambaForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "state-spaces/mamba-790m-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = MambaForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
NL = len(model.backbone.layers)
print("layers", NL)
print("주의: 'fast path is not available ... sequential implementation' 경고는 정상 = slow path 확인")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.17G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Loading weights:   0%|          | 0/482 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

layers 48
주의: 'fast path is not available ... sequential implementation' 경고는 정상 = slow path 확인


In [4]:
# [Δ 훅] dt_proj 출력(pre-softplus)에 위치별 gain. _CTRL로 제어.
_CTRL = {'gain_vec': None, 'layers': None}
def _mk_dt_hook(idx):
    def hook(mod, inp, out):
        g = _CTRL['gain_vec']
        if g is None: return out
        if _CTRL['layers'] is not None and idx not in _CTRL['layers']: return out
        L = out.shape[1]
        gv = g[:L].to(out.dtype).to(out.device)
        return out * gv.view(1, L, 1)
    return hook
_dt_handles = []
def setup_dt_hooks():
    global _dt_handles
    for h in _dt_handles: h.remove()
    _dt_handles = [l.mixer.dt_proj.register_forward_hook(_mk_dt_hook(i))
                   for i, l in enumerate(model.backbone.layers)]
def verify_dt_hooks():
    ids = tokenizer(" a b c d e f", return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad(): base = model(ids).logits[0, -1].clone()
    _CTRL['gain_vec'] = torch.full((ids.shape[1],), 0.5); _CTRL['layers'] = None
    with torch.no_grad(): pert = model(ids).logits[0, -1]
    _CTRL['gain_vec'] = None
    d = (base - pert).abs().max().item()
    return d > 1e-4, d
setup_dt_hooks()
ok, d = verify_dt_hooks(); print("Δ hook fires:", ok, "| max logit change", round(d, 3))

Δ hook fires: True | max logit change 106.302


In [5]:
# [진단] 레이어별 평균 decay A (-exp(A_log)). A<0 → recency 편향.
_A = [(-torch.exp(l.mixer.A_log)).mean().item() for l in model.backbone.layers]
print("mean A: L0", round(_A[0],3), "| Lmid", round(_A[NL//2],3), "| Llast", round(_A[-1],3))

mean A: L0 -33.752 | Lmid -42.991 | Llast -25.894


In [6]:
# [데이터] single-token 단어 풀 + MQAR (각 단어=1토큰 → 위치 정확)
def single_token_words(n, seed=0):
    rng = random.Random(seed); ids = list(range(1000, 45000)); rng.shuffle(ids); out = []
    for i in ids:
        s = tokenizer.decode([i]).strip()
        if (s.isascii() and s.isalpha() and 3 <= len(s) <= 8
                and len(tokenizer(" " + s, add_special_tokens=False).input_ids) == 1):
            out.append(s)
        if len(out) >= n: break
    return out
_WORDS = single_token_words(4000); print("single-token words:", len(_WORDS))

def _enc(word):
    return tokenizer(" " + word, add_special_tokens=False).input_ids

def build_mqar(N, rng, query_pair=0):
    ws = rng.sample(_WORDS, 2 * N)
    pairs = [(ws[2*j], ws[2*j+1]) for j in range(N)]
    seq = []; pos = []
    for (k, v) in pairs:
        kp = len(seq); seq += _enc(k)
        vp = len(seq); seq += _enc(v); pos.append((kp, vp))
    seq += tokenizer(" ?", add_special_tokens=False).input_ids
    seq += _enc(pairs[query_pair][0])
    ids = torch.tensor([seq], device=DEVICE)
    tgt = _enc(pairs[query_pair][1])[0]
    return ids, pos, tgt, pairs

def _recall_once(ids, tgt):
    with torch.no_grad(): lg = model(ids).logits[0, -1]
    return int(lg.argmax().item() == tgt)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


single-token words: 4000


In [7]:
# [용량 곡선] base recall vs N → N* 찾기
def base_recall(N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t)); h += _recall_once(ids, tgt)
    return h / n
print("[capacity curve]  (base가 ~0.4 되는 N이 N*)")
for N in [2, 8, 16, 24, 32, 48, 64]:
    print(f"  N={N:>3}: {base_recall(N, n=30):.2f}")

[capacity curve]  (base가 ~0.4 되는 N이 N*)
  N=  2: 0.63
  N=  8: 0.77
  N= 16: 0.77
  N= 24: 0.63
  N= 32: 0.53
  N= 48: 0.43
  N= 64: 0.33


In [8]:
# [레버 1: 생존/Δ] target value 이후(downstream) Δ를 og<1로 → target 보존
def trials_delta(N, og, layers=None, n=30, seed0=0, scope='downstream'):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        L = ids.shape[1]; v0 = pos[0][1]; gv = torch.ones(L)
        if scope == 'downstream':
            gv[v0 + 1:] = og
        elif scope == 'random':
            idxs = list(range(v0 + 1, L)); random.Random(seed0 + t + 999).shuffle(idxs)
            for j in idxs[:max(1, len(idxs)//2)]: gv[j] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n

N = 32   # 위 곡선 보고 base~0.4 되는 값으로
b = base_recall(N); print("base", round(b, 2))
print("[Gate1 dose] other-gain 스윕 (all layers)")
for og in [1.0, 0.9, 0.8, 0.7, 0.6, 0.4]:
    print(f"  og={og}: {trials_delta(N, og, n=30):.2f}")
print("[Gate1 specificity] downstream vs random (og=0.8)")
print(f"  downstream {trials_delta(N,0.8,scope='downstream',n=30):.2f} | random {trials_delta(N,0.8,scope='random',n=30):.2f}")

base 0.53
[Gate1 dose] other-gain 스윕 (all layers)
  og=1.0: 0.53
  og=0.9: 0.60
  og=0.8: 0.73
  og=0.7: 0.60
  og=0.6: 0.30
  og=0.4: 0.00
[Gate1 specificity] downstream vs random (og=0.8)
  downstream 0.73 | random 0.50


In [9]:
# [국소화] 어느 레이어 그룹이 회복을 옮기나 → L16-23 기대
groups = {f"L{a}-{a+7}": list(range(a, a + 8)) for a in range(0, NL, 8)}
base = base_recall(N)
print(f"[layer localization] og=0.8, base={base:.2f}")
for name, g in groups.items():
    r = trials_delta(N, 0.8, layers=g, n=30)
    print(f"  {name}: {r:.2f}  (Δ {r-base:+.2f})")

[layer localization] og=0.8, base=0.53
  L0-7: 0.53  (Δ +0.00)
  L8-15: 0.53  (Δ +0.00)
  L16-23: 0.67  (Δ +0.13)
  L24-31: 0.60  (Δ +0.07)
  L32-39: 0.53  (Δ +0.00)
  L40-47: 0.53  (Δ +0.00)


In [10]:
# [surgical vs full] L16-23만 vs 전층 + ppl guardrail
def ppl_under_gain(og, layers=None):
    txt = ("The transformer relies on attention while state space models compress "
           "history into a fixed hidden state for linear time inference.")
    ids = tokenizer(txt, return_tensors='pt').input_ids.to(DEVICE); L = ids.shape[1]
    _CTRL['gain_vec'] = torch.full((L,), og); _CTRL['layers'] = set(layers) if layers else None
    with torch.no_grad(): nll = model(ids, labels=ids).loss.item()
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return math.exp(nll)
base_ppl = ppl_under_gain(1.0); L1623 = list(range(16, 24))
print(f"base ppl {base_ppl:.1f}")
print(f"  full   og=0.8: recall {trials_delta(N,0.8,n=30):.2f}  ppl x{ppl_under_gain(0.8)/base_ppl:.2f}")
print(f"  L16-23 og=0.8: recall {trials_delta(N,0.8,layers=L1623,n=30):.2f}  ppl x{ppl_under_gain(0.8,L1623)/base_ppl:.2f}")

base ppl 232.2
  full   og=0.8: recall 0.73  ppl x1.81
  L16-23 og=0.8: recall 0.67  ppl x1.07


In [11]:
# [write-order = 암묵 우선순위] 같은 개입, 질의 쌍 위치만 바꿈
def recall_of_pair(N, j, og, layers=None, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, pairs = build_mqar(N, random.Random(seed0 + t), query_pair=j)
        L = ids.shape[1]; gv = torch.ones(L)
        if og < 1.0: gv[pos[j][1] + 1:] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n
print("[write-order priority] N=32, L16-23 og=0.8")
for j, lab in [(0, 'early(P1)'), (N//2, 'mid(P2)'), (N-1, 'late(P3)')]:
    bb = recall_of_pair(N, j, 1.0, n=30); aa = recall_of_pair(N, j, 0.8, layers=L1623, n=30)
    print(f"  {lab:>10} pair{j:>2}: base {bb:.2f} -> {aa:.2f}  (Δ {aa-bb:+.2f})")

[write-order priority] N=32, L16-23 og=0.8
   early(P1) pair 0: base 0.53 -> 0.67  (Δ +0.13)
     mid(P2) pair16: base 0.00 -> 0.10  (Δ +0.10)
    late(P3) pair31: base 0.03 -> 0.13  (Δ +0.10)


In [12]:
# [graded P1/P2/P3] 문서 시그니처를 *학습 없이* graded Δ로 실현 (탐색적)
# 각 value write의 보존을 그 쌍의 레벨 og로 근사. 단조 P1>P2>P3면 graded allocation 성립.
def graded_recall(N, og_lv, layers=None, n=30, seed0=0):
    res = {1: 0, 2: 0, 3: 0}; cnt = {1: 0, 2: 0, 3: 0}
    for t in range(n):
        rng = random.Random(seed0 + t); ws = rng.sample(_WORDS, 2 * N)
        pairs = [(ws[2*i], ws[2*i+1]) for i in range(N)]
        lvl = [(i % 3) + 1 for i in range(N)]
        seq = []; pos = []
        for (k, v) in pairs:
            kp = len(seq); seq += _enc(k)
            vp = len(seq); seq += _enc(v); pos.append((kp, vp))
        for Lq in [1, 2, 3]:
            j = lvl.index(Lq)
            s2 = seq + tokenizer(" ?", add_special_tokens=False).input_ids + _enc(pairs[j][0])
            ids = torch.tensor([s2], device=DEVICE); tgt = _enc(pairs[j][1])[0]
            gv = torch.ones(ids.shape[1])
            for i, (kp, vp) in enumerate(pos):
                gv[vp] = og_lv[lvl[i]]
            _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
            res[Lq] += _recall_once(ids, tgt); cnt[Lq] += 1
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return {k: res[k] / cnt[k] for k in [1, 2, 3]}
g = graded_recall(N, {1: 0.7, 2: 0.85, 3: 1.0}, layers=L1623, n=30)
print("[graded P1/P2/P3] L16-23")
print(f"  P1 {g[1]:.2f} | P2 {g[2]:.2f} | P3 {g[3]:.2f}")
print("  단조 P1>P2>P3 → graded allocation 성립 / 깨지면 → SSM uniform-decay 한계(C3)")

[graded P1/P2/P3] L16-23
  P1 0.47 | P2 0.13 | P3 0.07
  단조 P1>P2>P3 → graded allocation 성립 / 깨지면 → SSM uniform-decay 한계(C3)


In [13]:
# [레버 2: 정밀도/양자화] conv1d 출력(write 표현)을 선택 위치에서 k-bit
def fake_quant_vec(v, bits):
    if bits >= 16: return v
    qmax = (2 ** bits) - 1; lo = v.amin(-1, keepdim=True); hi = v.amax(-1, keepdim=True)
    scale = (hi - lo).clamp(min=1e-8) / qmax
    return torch.round((v - lo) / scale) * scale + lo
_QMASK = {'pos': None, 'bits': 16, 'fired': 0}
def _qhook(mod, inp, out):
    if _QMASK['bits'] >= 16 or not _QMASK['pos']: return out
    L = out.shape[-1]; pos = [p for p in _QMASK['pos'] if 0 <= p < L]
    if not pos: return out
    o = out.clone()
    for p in pos: o[:, :, p] = fake_quant_vec(o[:, :, p], _QMASK['bits'])
    _QMASK['fired'] += 1; return o
_qh = [l.mixer.conv1d.register_forward_hook(_qhook) for l in model.backbone.layers]

def rd_target(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        _QMASK['pos'] = list(pos[0]); _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
def rd_distractor(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        dp = [p for pr in pos[1:] for p in pr]
        _QMASK['pos'] = dp; _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
print("[A] target k-bit -> recall (정밀도-왜곡 곡선)")
for bt in [16, 8, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_target(bt, N, n=30):.2f}")
print("[B] 저우선 k-bit -> target recall (중요도 배분)")
for bt in [16, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_distractor(bt, N, n=30):.2f}")

[A] target k-bit -> recall (정밀도-왜곡 곡선)
  16-bit: 0.53
   8-bit: 0.53
   4-bit: 0.03
   2-bit: 0.00
   1-bit: 0.00
[B] 저우선 k-bit -> target recall (중요도 배분)
  16-bit: 0.53
   4-bit: 0.00
   2-bit: 0.00
   1-bit: 0.00


In [14]:
# [실험 A-1] N-sweep localization: 서로 다른 capacity pressure에서도 L16-23가 최적 circuit인가?
# N=16(낮은 압력), N=32(기준), N=64(높은 압력) 각각에서 8-layer band별 recall gain을 확인.
# → circuit 위치가 N에 무관하게 L16-23에 고정되면 'dedicated mechanism' 주장이 강화됨.

sweep_Ns   = [16, 32, 64]
band_names = [f'L{a}-{a+7}' for a in range(0, NL, 8)]
band_layers = [list(range(a, a + 8)) for a in range(0, NL, 8)]

print('[실험 A-1] 8-layer band localization across N  (og=0.8, n=30)')
header = f"{'N':>4} | {'base':>5} | " + ' | '.join(f'{b:>9}' for b in band_names)
print(header)
print('-' * len(header))

nsweep_loc = {}
for N in sweep_Ns:
    n_trials = 20 if N == 64 else 30   # N=64는 샘플 줄여 속도 확보
    b = base_recall(N, n=n_trials)
    band_r = [trials_delta(N, 0.8, layers=bl, n=n_trials) for bl in band_layers]
    nsweep_loc[N] = {'base': b, 'bands': dict(zip(band_names, band_r))}
    cols = ' | '.join(f'{r:.2f}({r-b:+.2f})' for r in band_r)
    print(f'{N:>4} | {b:>5.2f} | {cols}')

print('\n★ 각 N에서 가장 큰 gain을 보인 band:')
for N, res in nsweep_loc.items():
    best = max(res['bands'], key=res['bands'].get)
    print(f'  N={N:>2}: {best}  recall={res["bands"][best]:.2f}  '
          f'gain={res["bands"][best]-res["base"]:+.2f}')

[실험 A-1] 8-layer band localization across N  (og=0.8, n=30)
   N |  base |      L0-7 |     L8-15 |    L16-23 |    L24-31 |    L32-39 |    L40-47
------------------------------------------------------------------------------------
  16 |  0.77 | 0.73(-0.03) | 0.70(-0.07) | 0.83(+0.07) | 0.80(+0.03) | 0.80(+0.03) | 0.77(+0.00)
  32 |  0.53 | 0.53(+0.00) | 0.53(+0.00) | 0.67(+0.13) | 0.60(+0.07) | 0.53(+0.00) | 0.53(+0.00)
  64 |  0.40 | 0.40(+0.00) | 0.20(-0.20) | 0.60(+0.20) | 0.45(+0.05) | 0.15(-0.25) | 0.40(+0.00)

★ 각 N에서 가장 큰 gain을 보인 band:
  N=16: L16-23  recall=0.83  gain=+0.07
  N=32: L16-23  recall=0.67  gain=+0.13
  N=64: L16-23  recall=0.60  gain=+0.20


In [15]:
# [실험 A-2] N-sweep surgical table: 서로 다른 N에서 surgical L16-23 vs all-layer 비교
# recall gain과 PPL cost가 압력 수준에 걸쳐 일관되는지 확인.
# → PPL은 고정 텍스트 기준으로 N에 무관하나, recall gain은 N 의존 → 표로 분리.

print('[실험 A-2] Surgical (L16-23) vs Full — recall across N  (og=0.8)')
print(f"{'N':>4} | {'base':>6} | {'full':>6} | {'Δfull':>6} | {'surg':>6} | {'Δsurg':>6}")
print('-' * 48)

for N in sweep_Ns:
    n_trials = 20 if N == 64 else 30
    base_r = base_recall(N, n=n_trials)
    full_r  = trials_delta(N, 0.8, n=n_trials)
    surg_r  = trials_delta(N, 0.8, layers=L1623, n=n_trials)
    print(f'{N:>4} | {base_r:>6.2f} | {full_r:>6.2f} | {full_r-base_r:>+6.2f} | '
          f'{surg_r:>6.2f} | {surg_r-base_r:>+6.2f}')

# PPL은 N과 무관 — 고정 텍스트에서 한 번만 측정
_base_ppl = ppl_under_gain(1.0)
_full_ppl = ppl_under_gain(0.8)
_surg_ppl = ppl_under_gain(0.8, L1623)
print(f'\nPPL (N-independent, fixed passage): '
      f'base={_base_ppl:.1f}  '
      f'full=×{_full_ppl/_base_ppl:.2f}  '
      f'surg(L16-23)=×{_surg_ppl/_base_ppl:.2f}')
print('→ surgical PPL cost는 N과 무관 — 언어 모델링 비용은 circuit 선택에만 의존')

[실험 A-2] Surgical (L16-23) vs Full — recall across N  (og=0.8)
   N |   base |   full |  Δfull |   surg |  Δsurg
------------------------------------------------
  16 |   0.77 |   0.90 |  +0.13 |   0.83 |  +0.07
  32 |   0.53 |   0.73 |  +0.20 |   0.67 |  +0.13
  64 |   0.40 |   0.55 |  +0.15 |   0.60 |  +0.20

PPL (N-independent, fixed passage): base=232.2  full=×1.81  surg(L16-23)=×1.07
→ surgical PPL cost는 N과 무관 — 언어 모델링 비용은 circuit 선택에만 의존


## 읽는 법 / 정직한 스코프
- **용량곡선**: base가 ~0.4 되는 N을 `N`에 넣어라(셀 7 이후).
- **Gate1**: og↓이 recall↑(특정 구간), downstream≫random이면 특이성 ✓.
- **국소화**: L16-23가 양수 피크면 회로 확인. **L16-23만으로 전층과 같은 회복 + ppl×1.00**이 핵심.
- **write-order**: early(P1) Δ가 late(P3)보다 크면 "먼저 쓴 = 암묵 우선순위".
- **graded P1/P2/P3**: 단조면 graded allocation, 깨지면 SSM uniform-decay 한계(둘 다 보고가치).
- **양자화 레버**: A 단조하락 = 정밀도 제어 성립, B target↑ = 비트 재배분 성립.

**한계(반드시 보고):** 효과는 790m 합성 MQAR에서 견고하나 **1.4b 비재현·자연어 fragile**.
이는 회로가 모델·태스크 특이적임을 뜻하며, 현상의 *경계*로 정직하게 기술한다.